### Phase I Work Report: Data Extraction and Cryptographic Decoding

The goal of this phase was to bypass traditional, rate-limited Ethereum RPC nodes and extract the raw, unadulterated execution history of the Uniswap V3 ecosystem. Because smart contracts store data in compiled hexadecimal formats, the secondary objective was to cryptographically decode these logs into human-readable data structures (**Dataset A** and **Dataset B**) suitable for quantitative analysis.

**Methodology:**

1. **High-Throughput Extraction:**  
   I implemented the `hypersync` Parquet client to query the blockchain directly at the consensus layer. This bypassed the severe bottlenecks of standard `web3.py` API calls, allowing the pipeline to download millions of transaction receipts and block headers in minutes.

2. **Event Signature Targeting:**  
   The query was strictly constrained to the cryptographic Keccak-256 signatures for Uniswap V3's primary events: `Swap`, `Mint` (liquidity provision), and `Burn` (liquidity withdrawal).

3. **ABI Decoding:**  
   Smart contract logs are emitted in raw byte-code. I engineered a decoding matrix using `eth_abi` to unpack the 256-bit hexadecimal strings into standardized integers, isolating vital parameters such as `amount0`, `amount1`, `sqrtPriceX96`, and `tick`.

4. **Data Structuring:**  
   The decoded data was mapped to exact block timestamps, transaction hashes, and wallet sender addresses, creating a flawless, sequential ledger of every executed trade.


In [5]:
# impot what we need
import os
import asyncio
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta
import hypersync
from hypersync import (
    HypersyncClient,
    ClientConfig,
    Query,
    LogSelection,
    FieldSelection,
    LogField,
    BlockField,
    TransactionField,
    StreamConfig,
    HexOutput,
)
from config import OUT #to set the directory
import polars as pl
import math
from pathlib import Path
import shutil
import os

In [7]:
print(f"Saving data to: {OUT}")

Saving data to: C:\Users\Pouyan\python\thesis\Proposal\FINAL\Thesis_Output


---

## Raw Swap Extraction via HyperSync

This cell handles the massive extraction of raw Uniswap V3 swap logs. I breaked the 18 month timeframe into 3 month chunks to prevent memory overload, also features a built in checkpoint system (hs_check_...txt); if our internet drops or the kernel crashes, running this cell again will pick up exactly at the block where it failed.

In [9]:
# setting things up
API_TOKEN = "HyperSync Token"   # put HyperSync token here

# time window
START = datetime(2025, 1, 1, tzinfo=timezone.utc)
END   = datetime(2026, 7, 1, tzinfo=timezone.utc)

# Rough Ethereum mainnet anchors
ANCHOR_DATE  = datetime(2025, 1, 1, tzinfo=timezone.utc)
ANCHOR_BLOCK = 21_500_000
BLOCKS_PER_DAY = 7200

# The Keccak 256 hash for the Uniswap V3 "Swap" event
SWAP_TOPIC0 = "0xc42079f94a6350d7e6235f29174924f928cc2ac818eb64fed8004e115fbcca67"
# Pull 200,000 blocks per parquet file to keep file sizes manageable
BLOCK_BATCH = 200_000

# HELPER FUNCTIONS
def date_to_block(dt: datetime) -> int:
    days = (dt - ANCHOR_DATE).total_seconds() / 86400
    return int(ANCHOR_BLOCK + days * BLOCKS_PER_DAY)

def build_ranges():
    ranges = []
    cur = START
    while cur < END:
        nxt = min(cur + relativedelta(months=3), END)
        ranges.append((cur, nxt))
        cur = nxt
    return ranges

# MAIN EXTRACTION LOOP
async def main():
    client = HypersyncClient(
        ClientConfig(
            url="https://eth.hypersync.xyz",
            bearer_token=API_TOKEN,
        )
    )
    
    # Define exactly which columns we want from the blockchain to save bandwidth
    field_selection = FieldSelection(
        block=[BlockField.NUMBER, BlockField.TIMESTAMP],
        transaction=[TransactionField.HASH, TransactionField.FROM],
        log=[
            LogField.BLOCK_NUMBER,
            LogField.LOG_INDEX,
            LogField.TRANSACTION_INDEX,
            LogField.TRANSACTION_HASH,
            LogField.ADDRESS,
            LogField.TOPIC0,
            LogField.TOPIC1,
            LogField.TOPIC2,
            LogField.DATA,
        ],
    )

    config = StreamConfig(
        hex_output=HexOutput.PREFIXED,
        event_signature=(
            "Swap(address indexed sender, address indexed recipient, "
            "int256 amount0, int256 amount1, uint160 sqrtPriceX96, "
            "uint128 liquidity, int24 tick)"
        ),
    )

    ranges = build_ranges()
    print("Chunks:")
    for s, e in ranges:
        print(f"  {s.date()} → {e.date()}  "
              f"(blocks ~{date_to_block(s):,} → ~{date_to_block(e):,})")
     # Iterate through each 3 month period
    for start_dt, end_dt in ranges:
        chunk_name = f"{start_dt.date()}_to_{end_dt.date()}"
        out_dir = f"uniswap_v3_swaps_{chunk_name}"
        check_file = f"hs_check_{chunk_name}.txt"

        start_block = date_to_block(start_dt)
        end_block   = date_to_block(end_dt)

        # Checkpoint logic: Resume where we left off if interrupted
        if os.path.exists(check_file):
            with open(check_file) as f:
                current = int(f.read().strip())
            print(f"\n[{chunk_name}] Resuming from block {current:,}")
        else:
            current = start_block
            print(f"\n[{chunk_name}] Starting from block {current:,}")

        os.makedirs(out_dir, exist_ok=True)

        # Stream the data in batches of 200,000 blocks
        while current < end_block:
            batch_end = min(current + BLOCK_BATCH, end_block)
            part_name = f"{out_dir}/blocks_{current}_{batch_end}"

            print(f"  Pulling blocks {current:,} → {batch_end:,} ...")

            query = Query(
                from_block=current,
                to_block=batch_end,
                logs=[
                    LogSelection(
                        topics=[[SWAP_TOPIC0]],
                    )
                ],
                field_selection=field_selection,
            )

            try:
                await client.collect_parquet(part_name, query, config)
            except Exception as e:
                print(f"  Error: {e}")
                print("  Checkpoint saved. Re-run to resume.")
                break

            # Update checkpoint after a successful batch
            current = batch_end
            with open(check_file, "w") as f:
                f.write(str(current))
            print(f"  Done. Checkpoint → {current:,}")

        else:
            print(f"[{chunk_name}] COMPLETE")

    print("\nAll chunks finished (or paused).")

await main()

Chunks:
  2025-01-01 → 2025-04-01  (blocks ~21,500,000 → ~22,148,000)
  2025-04-01 → 2025-07-01  (blocks ~22,148,000 → ~22,803,200)
  2025-07-01 → 2025-10-01  (blocks ~22,803,200 → ~23,465,600)
  2025-10-01 → 2026-01-01  (blocks ~23,465,600 → ~24,128,000)
  2026-01-01 → 2026-04-01  (blocks ~24,128,000 → ~24,776,000)
  2026-04-01 → 2026-07-01  (blocks ~24,776,000 → ~25,431,200)

[2025-01-01_to_2025-04-01] Starting from block 21,500,000
  Pulling blocks 21,500,000 → 21,700,000 ...
  Done. Checkpoint → 21,700,000
  Pulling blocks 21,700,000 → 21,900,000 ...
  Done. Checkpoint → 21,900,000
  Pulling blocks 21,900,000 → 22,100,000 ...
  Done. Checkpoint → 22,100,000
  Pulling blocks 22,100,000 → 22,148,000 ...
  Done. Checkpoint → 22,148,000
[2025-01-01_to_2025-04-01] COMPLETE

[2025-04-01_to_2025-07-01] Starting from block 22,148,000
  Pulling blocks 22,148,000 → 22,348,000 ...
  Done. Checkpoint → 22,348,000
  Pulling blocks 22,348,000 → 22,548,000 ...
  Done. Checkpoint → 22,548,000
  Pu

---

## Extracting the Pool Registry (Factory Logs)
Uniswap V3 is not one single contract; it is thousands of individual pool contracts generated by a central “Factory” contract. This cell listens exclusively to the Factory contract to map out every single liquidity pool ever created on the network.

In [20]:
# The canonical Uniswap V3 Factory address on Ethereum Mainnet
FACTORY = "0x1F98431c8aD98523631AE4a59f267346ea31F984"  # Uniswap V3 Factory, mainnet

async def pull_pools():
    # Setup client specifically for this query
    client = HypersyncClient(ClientConfig(
        url="https://eth.hypersync.xyz",
        bearer_token=API_TOKEN, # Reusing the token from Cell 1
    ))
    # Query logs from the exact block the Factory was deployed (12,369,621)
    q = Query(
        from_block=12_369_621,          # factory deployment
        to_block=25_431_200,
        logs=[LogSelection(address=[FACTORY])],   # filter by address, no topic needed
        field_selection=FieldSelection(
            log=[LogField.BLOCK_NUMBER, LogField.LOG_INDEX,
                 LogField.ADDRESS, LogField.TOPIC0, LogField.TOPIC1,
                 LogField.TOPIC2, LogField.TOPIC3, LogField.DATA],
        ),
    )
    cfg = StreamConfig(
        hex_output=HexOutput.PREFIXED,
        event_signature=(
            "PoolCreated(address indexed token0, address indexed token1, "
            "uint24 indexed fee, int24 tickSpacing, address pool)"
        ),
    )
     # Save to a dedicated pool folder
    await client.collect_parquet("uniswap_v3_pools", q, cfg)

await pull_pools()

---

## Parsing and Cleaning the Pool Registry
The raw logs generated in Cell 2 are ugly hexadecimal strings. This cell uses Polars string slicing to extract the underlying Ethereum addresses for Token0, Token1, the Pool itself, and the fee structures.

In [23]:
# Read the raw factory logs
lg = pl.read_parquet("uniswap_v3_pools/logs.parquet")
print(lg.shape, lg.columns)
print(lg["topic0"].value_counts(sort=True).head(5))

# Identify the 'PoolCreated' event signature dynamically
pc_topic = lg["topic0"].value_counts(sort=True).row(0)[0]
pc = lg.filter(pl.col("topic0") == pc_topic)
print(f"PoolCreated logs: {len(pc):,} of {len(lg):,}")

# 3. Hex String Slicing Logic:
# EVM logs pad data to 32 bytes (64 hex characters).
# Topic 1 & 2: Token addresses (Take the last 40 chars and add '0x')
# Topic 3: Fee tier (Take the last 6 chars and parse as integer)
# Data: word0 (tickSpacing) and word1 (pool address)
pools = pc.select(
    ("0x" + pl.col("topic1").str.slice(-40)).str.to_lowercase().alias("token0"),
    ("0x" + pl.col("topic2").str.slice(-40)).str.to_lowercase().alias("token1"),
    pl.col("topic3").str.slice(-6).str.to_integer(base=16, strict=False).alias("fee"),
    pl.col("data").str.slice(60, 6).str.to_integer(base=16, strict=False).alias("tick_spacing"),
    ("0x" + pl.col("data").str.slice(90, 40)).str.to_lowercase().alias("pool"),
).unique(subset="pool")

# Save the clean registry for later datasets
pools.write_parquet("uniswap_v3_pools_clean.parquet")
print(pools.shape)
print(pools["fee"].value_counts(sort=True))
print(pools.head(5))

(66335, 8) ['log_index', 'block_number', 'address', 'data', 'topic0', 'topic1', 'topic2', 'topic3']
shape: (3, 2)
┌─────────────────────────────────┬───────┐
│ topic0                          ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ 0x783cca1c0412dd0d695e784568c9… ┆ 66327 │
│ 0xb532073b38c83145e3e5135377a0… ┆ 4     │
│ 0xc66a3fdf07232cdd185febcc6579… ┆ 4     │
└─────────────────────────────────┴───────┘
PoolCreated logs: 66,327 of 66,335
(66327, 5)
shape: (4, 2)
┌───────┬───────┐
│ fee   ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 10000 ┆ 34384 │
│ 3000  ┆ 21141 │
│ 100   ┆ 6120  │
│ 500   ┆ 4682  │
└───────┴───────┘
shape: (5, 5)
┌─────────────────────────┬────────────────────────┬───────┬──────────────┬────────────────────────┐
│ token0                  ┆ token1                 ┆ fee   ┆ tick_spacing ┆ pool                   │
│ ---                     ┆ ---   

---

## The Unified Builder (Hex Decoder & Data Cleaner)

This is the most computationally complex cell in this notebook and in my pipeline in general. Blockchain numeric values (like swap amounts and square root prices) are stored as 256 bit signed integers. Since Python/Polars maxes out at 64 bit natively (or 128 bit with some effort), this cell defines custom _limbs functions to carefully chunk and decode the hex strings without losing precision or overflowing.

In [26]:
# The script will output the decoded, joined files here
OUTDIR = Path("clean"); OUTDIR.mkdir(exist_ok=True)
BADDIR = Path("clean_quarantine"); BADDIR.mkdir(exist_ok=True)
MAX_TICK = 887272
L32      = pl.lit(2**32, dtype=pl.Int128)
LOG_1P   = math.log(1.0001)

# Hex Decoders (EVM 256-BIT HEX DECODERS)
def _word(col):
    return (pl.col(col).cast(pl.Utf8).str.to_lowercase()
              .str.strip_chars_start("0x").str.pad_start(64, "0"))

def _limbs(w, n_hex):
    off = 64 - n_hex
    return [w.str.slice(off + i * 8, 8).str.to_integer(base=16, strict=False)
            for i in range(n_hex // 8)]

def i256_f64(col, alias):
    L = _limbs(_word(col), 64)
    v = pl.when(L[0] >= 2**31).then(L[0] - 2**32).otherwise(L[0]).cast(pl.Float64)
    for l in L[1:]:
        v = v * 4294967296.0 + l.cast(pl.Float64)
    return v.alias(alias)

def i256_exact(col, alias):
    w   = _word(col)
    hi  = w.str.slice(0, 32)
    neg = w.str.slice(32, 1).is_in(list("89abcdef"))
    fits = pl.when(neg).then(hi == "f" * 32).otherwise(hi == "0" * 32)
    L = _limbs(w, 32)
    v = pl.when(L[0] >= 2**31).then(L[0] - 2**32).otherwise(L[0]).cast(pl.Int128)
    for l in L[1:]:
        v = v * L32 + l.cast(pl.Int128)
    return pl.when(fits).then(v).otherwise(None).alias(alias)

def u_f64(col, alias, n_hex=64):
    L = _limbs(_word(col), n_hex)
    v = L[0].cast(pl.Float64)
    for l in L[1:]:
        v = v * 4294967296.0 + l.cast(pl.Float64)
    return v.alias(alias)

def u128_exact(col, alias):
    w    = _word(col)
    fits = (w.str.slice(0, 32) == "0" * 32) & w.str.slice(32, 1).is_in(list("01234567"))
    L = _limbs(w, 32)
    v = L[0].cast(pl.Int128)
    for l in L[1:]:
        v = v * L32 + l.cast(pl.Int128)
    return pl.when(fits).then(v).otherwise(None).alias(alias)

def i24(col, alias):
    u = _word(col).str.slice(58, 6).str.to_integer(base=16, strict=False).cast(pl.Int32)
    return pl.when(u >= 2**23).then(u - 2**24).otherwise(u).alias(alias)

# MAIN BATCH PROCESSING FUNCTION
def clean_batch(d: Path, out_path: Path) -> dict:
    logs = pl.read_parquet(d / "logs.parquet",
        columns=["block_number", "log_index", "transaction_index",
                 "transaction_hash", "address"]).rename({"address": "pool_address"})
    dec = pl.read_parquet(d / "decoded_logs.parquet")

    if len(logs) != len(dec):
        raise RuntimeError(f"MISALIGNED {d}: {len(logs):,} vs {len(dec):,}")

    df = logs.hstack(dec.rename({c: f"d_{c}" for c in dec.columns}))
    d_cols = [c for c in df.columns if c.startswith("d_")]

    # Apply decoders
    df = df.with_columns([
        pl.col("pool_address").str.to_lowercase(),
        pl.col("d_sender").str.to_lowercase().alias("sender"),
        pl.col("d_recipient").str.to_lowercase().alias("recipient"),
        i256_f64  ("d_amount0",      "amount0_f64"),
        i256_f64  ("d_amount1",      "amount1_f64"),
        i256_exact("d_amount0",      "amount0"),
        i256_exact("d_amount1",      "amount1"),
        u_f64     ("d_sqrtPriceX96", "sqrt_price_x96", 40),
        u_f64     ("d_liquidity",    "liquidity_f64", 32),
        u128_exact("d_liquidity",    "liquidity"),
        i24       ("d_tick",         "tick"),
    ]).with_columns(
        (pl.col("amount0").is_null() | pl.col("amount1").is_null()).alias("amount_overflow")
    ).with_columns(
        # Validate price integrity by comparing reported tick to mathematically implied tick
        (2.0 * (pl.col("sqrt_price_x96") / 2.0**96).log() / LOG_1P).alias("_implied_tick")
    ).with_columns(
        ((pl.col("_implied_tick") - pl.col("tick").cast(pl.Float64)).abs() > 2
         ).fill_null(True).alias("_price_bad")
    )

    # guards against corrupted data
    a0, a1 = pl.col("amount0_f64"), pl.col("amount1_f64")
    chk = df.select(
        n              = pl.len(),
        nulls_f64      = (a0.is_null() | a1.is_null()).sum(),
        both_pos       = ((a0 > 0) & (a1 > 0)).sum(),
        both_neg       = ((a0 < 0) & (a1 < 0)).sum(),
        zero_leg       = ((a0 == 0) | (a1 == 0)).sum(),
        overflow       = pl.col("amount_overflow").sum(),
        neg_liq        = (pl.col("liquidity_f64") < 0).sum(),
        bad_tick       = (pl.col("tick").abs() > MAX_TICK).sum(),
        zero_price     = (pl.col("sqrt_price_x96") <= 0).sum(),
        price_mismatch = pl.col("_price_bad").sum(),
    ).row(0, named=True)

    for k in ("nulls_f64", "both_pos", "both_neg", "neg_liq", "bad_tick"):
        if chk[k]:
            raise RuntimeError(f"DECODE FAIL {d.name} [{k}]: {chk}")

    # QUARANTINE LOGIC (For minor price mismatches)
    PRICE_TOL = max(20, int(2e-5 * chk["n"]))          # 20 rows, or 0.002%
    if chk["price_mismatch"]:
        (df.filter(pl.col("_price_bad"))
           .write_parquet(BADDIR / f"{out_path.stem}__price.parquet",
                          compression="zstd"))
        if chk["price_mismatch"] > PRICE_TOL:
            raise RuntimeError(f"PRICE FAIL {d.name} "
                               f"({chk['price_mismatch']:,} > tol {PRICE_TOL:,}): {chk}")
        print(f"  warn {d.name}: price_mismatch={chk['price_mismatch']:,} "
              f"(tol {PRICE_TOL:,}) → quarantined, kept in output")

    df = df.drop(d_cols + ["_implied_tick", "_price_bad"])

    # JOIN BLOCK TIMESTAMPS
    blocks = pl.read_parquet(d / "blocks.parquet").rename(
        {"number": "block_number", "timestamp": "ts_raw"})
    if blocks["ts_raw"].dtype == pl.String:
        blocks = blocks.with_columns(
            pl.col("ts_raw").str.strip_chars_start("0x")
              .str.to_integer(base=16, strict=False).alias("ts_raw"))
    blocks = blocks.select(
        "block_number",
        pl.from_epoch(pl.col("ts_raw").cast(pl.Int64), time_unit="s").alias("block_time"))

    # JOIN TRANSACTION ORIGINS ('FROM' WALLET)
    tx_names = pl.scan_parquet(d / "transactions.parquet").collect_schema().names()
    has_to   = "to" in tx_names
    txs = (pl.read_parquet(d / "transactions.parquet",
                           columns=["hash", "from"] + (["to"] if has_to else []))
             .rename({"hash": "transaction_hash", "from": "tx_from",
                      **({"to": "tx_to"} if has_to else {})})
             .with_columns([pl.col(c).str.to_lowercase()
                            for c in ["tx_from"] + (["tx_to"] if has_to else [])])
             .unique(subset="transaction_hash"))

    # Final Join & Sort
    df = (df.join(blocks, on="block_number", how="left")
            .join(txs,    on="transaction_hash", how="left")
            .sort(["block_number", "log_index"]))

    miss = df.select(no_time=pl.col("block_time").is_null().sum(),
                     no_from=pl.col("tx_from").is_null().sum()).row(0, named=True)
    if miss["no_time"]:
        raise RuntimeError(f"BLOCK JOIN FAIL {d.name}: {miss}")

    # Write output successfully
    df.write_parquet(out_path, compression="zstd")
    chk["batch"] = out_path.stem
    return chk

# Resume Loop
batch_dirs = sorted(p.parent for p in Path(".").glob("uniswap_v3_swaps_*/*/logs.parquet"))
print(f"found {len(batch_dirs)} batch dirs")
rows = []

for d in batch_dirs:
    out = OUTDIR / f"{d.parent.name}__{d.name}.parquet"
    if out.exists():
        print(f"skip {out.name}"); continue
    c = clean_batch(d, out)
    rows.append(c)
    print(f"{out.name}: {c['n']:,} rows  overflow={c['overflow']:,}  zero_leg={c['zero_leg']:,}")

# Generate a final statistical report
if rows:
    rep = pl.DataFrame(rows).select("batch", "n", "overflow", "zero_leg", "zero_price", "price_mismatch")
    print(rep)
    print("NEW ROWS", f"{rep['n'].sum():,}")
    

found 24 batch dirs
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21500000_21700000.parquet: 2,586,295 rows  overflow=2  zero_leg=82
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21700000_21900000.parquet: 3,255,821 rows  overflow=0  zero_leg=81
  warn blocks_21900000_22100000: price_mismatch=4 (tol 68) → quarantined, kept in output
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21900000_22100000.parquet: 3,438,146 rows  overflow=6  zero_leg=118
  warn blocks_22100000_22148000: price_mismatch=2 (tol 20) → quarantined, kept in output
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_22100000_22148000.parquet: 662,532 rows  overflow=1  zero_leg=26
  warn blocks_22148000_22348000: price_mismatch=12 (tol 64) → quarantined, kept in output
uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22148000_22348000.parquet: 3,245,002 rows  overflow=5  zero_leg=90
uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22348000_22548000.parquet: 3,065,328 rows  overflow=9  zero_leg=95
uniswap_v3_

---
**Results and Data Integrity:**

This phase successfully generated the baseline architecture: **Dataset A** (the raw ledger of executed swaps) and **Dataset B** (the ledger of liquidity provisioning). By utilizing out-of-core processing (`Polars`), the system efficiently handled data scales that would typically crash standard memory environments like `Pandas`.